# ComfyUI AI Influencer on Kaggle

This notebook sets up ComfyUI for AI influencer generation on Kaggle with T4 x2 GPUs.

### Instructions:
1. Enable T4 x2 GPU.
2. Turn on Internet Access.
3. Run cells sequentially.

In [ ]:
!git clone https://github.com/Dhruwkd/comfyui-kaggle-influencer.git
%cd comfyui-kaggle-influencer
!chmod +x scripts/*.sh
!./scripts/install_nodes.sh
!./scripts/download_models.sh

In [ ]:
# Install cloudflared binary to a local path (avoid "Text file busy" on /usr/local/bin)
!mkdir -p /kaggle/working/bin
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/bin/cloudflared
!chmod +x /kaggle/working/bin/cloudflared

# Also install localtunnel as fallback (more reliable on Kaggle)
!npm install -g localtunnel 2>/dev/null || echo "localtunnel install failed, will use cloudflared only"

import subprocess
import threading
import time
import os
import sys
import re
import urllib.request
import urllib.error

# Verify ComfyUI exists
comfyui_path = "/kaggle/working/ComfyUI"
if not os.path.exists(comfyui_path):
    raise RuntimeError(f"ComfyUI not found at {comfyui_path}. Did cell 1 complete successfully?")

os.chdir(comfyui_path)

# ─── FIRST: Start ComfyUI in background ───
print("🚀 Starting ComfyUI on port 8188...", flush=True)
comfy_proc = subprocess.Popen(
    [sys.executable, "main.py", "--lowvram", "--port", "8188", "--listen", "0.0.0.0"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Print ComfyUI startup logs in background
def print_comfy_logs():
    for line in comfy_proc.stdout:
        print(f"[ComfyUI] {line.strip()}", flush=True)

comfy_log_thread = threading.Thread(target=print_comfy_logs, daemon=True)
comfy_log_thread.start()

# Wait for ComfyUI to be ready locally
print("⏳ Waiting for ComfyUI to start locally...", flush=True)
local_ready = False
for attempt in range(30):
    try:
        req = urllib.request.Request("http://localhost:8188", headers={'User-Agent': 'Mozilla/5.0'})
        response = urllib.request.urlopen(req, timeout=5)
        if response.status == 200:
            local_ready = True
            print("✅ ComfyUI is ready locally!", flush=True)
            break
    except Exception:
        pass
    time.sleep(2)
    if attempt % 5 == 4:
        print(f"  Still waiting... ({attempt+1}/30)", flush=True)

if not local_ready:
    print("❌ ComfyUI failed to start. Check [ComfyUI] logs above.", flush=True)

# ─── SECOND: Start tunnel (now that ComfyUI is ready) ───
tunnel_url = {"value": None}
tunnel_type = {"value": None}
tunnel_ready = threading.Event()
tunnel_verified = threading.Event()

def verify_url(url, timeout=10):
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        response = urllib.request.urlopen(req, timeout=timeout)
        return response.status == 200
    except Exception:
        return False

def run_cloudflared():
    print("[cloudflared] Starting...", flush=True)
    proc = subprocess.Popen(
        ["/kaggle/working/bin/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8188"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in proc.stdout:
        line = line.strip()
        print(f"[cloudflared] {line}", flush=True)
        if "trycloudflare.com" in line and not tunnel_url["value"]:
            match = re.search(r'(https://[a-zA-Z0-9\-]+\.trycloudflare\.com)', line)
            if match:
                tunnel_url["value"] = match.group(1)
                tunnel_type["value"] = "cloudflared"
                tunnel_ready.set()

def run_localtunnel():
    print("[localtunnel] Starting...", flush=True)
    proc = subprocess.Popen(
        ["lt", "--port", "8188", "--host", "https://localtunnel.me"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in proc.stdout:
        line = line.strip()
        print(f"[localtunnel] {line}", flush=True)
        if "your url is:" in line.lower() and not tunnel_url["value"]:
            match = re.search(r'(https://[a-zA-Z0-9\-]+\.loca\.lt)', line)
            if match:
                tunnel_url["value"] = match.group(1)
                tunnel_type["value"] = "localtunnel"
                tunnel_ready.set()

if local_ready:
    # Try cloudflared first
    tunnel_thread = threading.Thread(target=run_cloudflared, daemon=True)
    tunnel_thread.start()
    print("⏳ Waiting for cloudflared tunnel URL...", flush=True)
    tunnel_ready.wait(timeout=30)

    # Fallback to localtunnel
    if not tunnel_url["value"]:
        print("⚠️ Cloudflared didn't produce a URL. Trying localtunnel...", flush=True)
        tunnel_ready.clear()
        tunnel_thread = threading.Thread(target=run_localtunnel, daemon=True)
        tunnel_thread.start()
        tunnel_ready.wait(timeout=30)

    if not tunnel_url["value"]:
        print("❌ Both tunnels failed. Check logs above.", flush=True)
    else:
        url = tunnel_url['value']
        tunnel_name = tunnel_type['value']
        print(f"\n🔗 {tunnel_name} URL detected: {url}", flush=True)
        print("⏳ Verifying tunnel is accessible...", flush=True)
        
        for attempt in range(20):
            if verify_url(url):
                print(f"✅ Tunnel verified and accessible!", flush=True)
                tunnel_verified.set()
                break
            else:
                print(f"  Attempt {attempt+1}/20: Not ready yet, waiting...", flush=True)
            time.sleep(2)
        
        if not tunnel_verified.is_set():
            print("⚠️ Tunnel not fully accessible yet. May need more time.", flush=True)

    # Display the URL prominently
    if tunnel_url["value"]:
        url = tunnel_url['value']
        tunnel_name = tunnel_type['value']
        print(f"\n{'#'*70}", flush=True)
        print(f"#  🌐  COMFYUI TUNNEL URL ({tunnel_name}) — COPY THIS LINK:", flush=True)
        print(f"#", flush=True)
        print(f"#      {url}", flush=True)
        print(f"#", flush=True)
        if tunnel_verified.is_set():
            print(f"#  ✅  TUNNEL VERIFIED — SHOULD WORK NOW", flush=True)
        else:
            print(f"#  ⏳  TUNNEL MAY NEED MORE TIME — WAIT 30s THEN TRY", flush=True)
        print(f"#  👆  OPEN THIS IN YOUR BROWSER (or Claude's Browser pane)", flush=True)
        print(f"#  ❌  DO NOT USE: http://localhost:8188  OR  http://0.0.0.0:8188", flush=True)
        print(f"{'#'*70}\n", flush=True)

# Keep ComfyUI running (this blocks)
print("🔄 ComfyUI running. Press Stop to shut down.", flush=True)
comfy_proc.wait()